# Production Recommender Engine: Latent Dirichlet Allocation (LDA) & Belief Propagation on Markov Random Fields (MRF)

This Jupyter Notebook holds the complete production-grade recommendation engine for **LeetPath**. By structure, the Flask backend reads and executes this notebook dynamically at startup, using it as the active recommendation module instead of `recommender.py`.

Here is the walkthrough of the algorithm components:

## 1. Import Dependencies & Utility Libraries

We load necessary computational and machine learning modules from Python's standard libraries and environment.

In [ ]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import logging

logger = logging.getLogger(__name__)
print("Libraries loaded successfully.")

## 2. Markov Random Field (MRF) Graph Model

We define the Pairwise MRF where nodes represent questions. Unary potentials set individual prior scores, and pairwise potentials model homophily coupling between similar nodes.

In [ ]:
class PairwiseMRF:
    """
    Markov Random Field Graph mapping unary and pairwise potentials.
    Each state value represents recommendation likelihood (1 = recommend, 0 = do not recommend).
    """
    def __init__(self, size):
        self.size = size
        self.unary_potentials = np.ones((size, 2))  # [node, state_val]
        self.adjacency_list = {i: [] for i in range(size)}
        self.edge_potentials = {}  # Key: (u, v), value: 2x2 matrix

    def set_unary(self, node_idx, p0, p1):
        self.unary_potentials[node_idx, 0] = p0
        self.unary_potentials[node_idx, 1] = p1
    
    def add_edge(self, u, v, weight):
        if v not in self.adjacency_list[u]:
            self.adjacency_list[u].append(v)
        if u not in self.adjacency_list[v]:
            self.adjacency_list[v].append(u)
        
        # Setup pairwise homophily matrix
        psi = np.array([
            [1.0 + weight, 1.0],
            [1.0, 1.0 + weight]
        ])
        self.edge_potentials[(u, v)] = psi
        self.edge_potentials[(v, u)] = psi

## 3. Sum-Product Belief Propagation

We implement the message-passing algorithm on the sparse MRF to compute the marginal belief of recommendations.

In [ ]:
class BeliefPropagation:
    """
    Sum-Product message-passing algorithm on pairwise MRF.
    """
    def __init__(self, mrf):
        self.mrf = mrf
        self.messages = {}
        for u, neighbors in mrf.adjacency_list.items():
            for v in neighbors:
                self.messages[(u, v)] = np.array([0.5, 0.5])

    def run_iterations(self, max_iter=3):
        for iteration in range(max_iter):
            new_messages = {}
            for (u, v) in self.messages.keys():
                incoming_prod = np.array([1.0, 1.0])
                for k in self.mrf.adjacency_list[u]:
                    if k != v:
                        incoming_prod *= self.messages.get((k, u), np.array([0.5, 0.5]))
                
                local_pot = self.mrf.unary_potentials[u] * incoming_prod
                psi_uv = self.mrf.edge_potentials[(u, v)]
                new_msg = np.zeros(2)
                for xv in range(2):
                    new_msg[xv] = np.sum(local_pot * psi_uv[:, xv])
                
                msg_sum = np.sum(new_msg)
                if msg_sum > 0:
                    new_msg /= msg_sum
                else:
                    new_msg = np.array([0.5, 0.5])
                
                new_messages[(u, v)] = new_msg
            self.messages = new_messages
            
    def compute_beliefs(self):
        beliefs = np.zeros((self.mrf.size, 2))
        for u in range(self.mrf.size):
            incoming_prod = np.array([1.0, 1.0])
            for k in self.mrf.adjacency_list[u]:
                incoming_prod *= self.messages.get((k, u), np.array([0.5, 0.5]))
            
            belief_u = self.mrf.unary_potentials[u] * incoming_prod
            b_sum = np.sum(belief_u)
            if b_sum > 0:
                beliefs[u] = belief_u / b_sum
            else:
                beliefs[u] = np.array([0.5, 0.5])
        return beliefs

## 4. Latent Dirichlet Allocation (LDA) using Collapsed Gibbs Sampling

We model the latent topics of question descriptions using a Gibbs sampler.

In [ ]:
class GibbsLDA:
    """
    Lightweight collapsed Gibbs sampling Latent Dirichlet Allocation (LDA).
    """
    def __init__(self, n_topics=4, alpha=0.1, beta=0.1, n_iter=10):
        self.n_topics = n_topics
        self.alpha = alpha
        self.beta = beta
        self.n_iter = n_iter

    def fit_transform(self, corpus):
        from collections import Counter
        words_per_doc = [str(doc).lower().split() for doc in corpus]
        all_words = [w for doc in words_per_doc for w in doc if len(w) > 3 and w.isalpha()]
        
        vocab_counter = Counter(all_words)
        frequent_words = [w for w, count in vocab_counter.most_common(500)]
        vocab = {word: idx for idx, word in enumerate(frequent_words)}
        
        N = len(corpus)
        V = len(vocab)
        if V == 0:
            return np.ones((N, self.n_topics)) / float(self.n_topics)
            
        docs = []
        for doc in words_per_doc:
            docs.append([vocab[w] for w in doc if w in vocab])
            
        n_d_k = np.zeros((N, self.n_topics))
        n_k_v = np.zeros((self.n_topics, V))
        n_k = np.zeros(self.n_topics)
        z_d_n = []
        
        for d in range(N):
            z_n = []
            for w in docs[d]:
                k = np.random.randint(self.n_topics)
                z_n.append(k)
                n_d_k[d, k] += 1
                n_k_v[k, w] += 1
                n_k[k] += 1
            z_d_n.append(np.array(z_n))
            
        for it in range(self.n_iter):
            for d in range(N):
                for n, w in enumerate(docs[d]):
                    k = z_d_n[d][n]
                    n_d_k[d, k] -= 1
                    n_k_v[k, w] -= 1
                    n_k[k] -= 1
                    
                    p_k = (n_d_k[d, :] + self.alpha) * (n_k_v[:, w] + self.beta) / (n_k + V * self.beta)
                    p_k_sum = np.sum(p_k)
                    if p_k_sum > 0:
                        p_k /= p_k_sum
                    else:
                        p_k = np.ones(self.n_topics) / float(self.n_topics)
                        
                    new_k = np.random.choice(self.n_topics, p=p_k)
                    z_d_n[d][n] = new_k
                    n_d_k[d, new_k] += 1
                    n_k_v[new_k, w] += 1
                    n_k[new_k] += 1
                    
        theta = (n_d_k + self.alpha) / (np.sum(n_d_k, axis=1, keepdims=True) + self.n_topics * self.alpha)
        return theta

## 5. Main QuestionRecommender Module

This brings everything together: calculating the TF-IDF cosine similarity, Latent topics, MRF edge-building, and calling Belief Propagation.

In [ ]:
class QuestionRecommender:
    """
    Unified LeetCode question recommender engine using MRF & Belief Propagation.
    """
    def __init__(self, file_path="data.json"):
        self.file_path = file_path
        self.df = self.load_and_preprocess_data(file_path)
        self.similarity_matrix = None
        self.topic_matrix = None
        self.potential_matrix = None
        self.G = None

        logger.info("Initializing similarity matrix...")
        self.calculate_similarity_matrix()
        logger.info("Initializing topic matrix...")
        self.calculate_topic_matrix()
        logger.info("Initializing potential matrix...")
        self.calculate_potential_matrix()
        logger.info("Building recommendation graph...")
        self.build_graph()
        logger.info("Recommender initialization complete.")

    def load_and_preprocess_data(self, file_path):
        df = pd.read_json(file_path)
        if 'titleSlug' not in df.columns and 'titleSlug' in df.index:
            df = df.T
        df = self.preprocess_data(df)
        return df

    def preprocess_data(self, df):
        scaler = MinMaxScaler()
        df['likability'] = pd.to_numeric(df['likability'], errors='coerce').fillna(50.0)
        df['accuracy'] = pd.to_numeric(df['accuracy'], errors='coerce').fillna(50.0)
        
        def map_difficulty(val):
            val_str = str(val).strip().lower()
            if val_str in ['1', 'easy']:
                return 'easy'
            elif val_str in ['3', 'hard']:
                return 'hard'
            else:
                return 'medium'
                
        df['difficulty'] = df['difficulty'].apply(map_difficulty)
        df[['likability_norm', 'accuracy_norm']] = scaler.fit_transform(df[['likability', 'accuracy']])
        return df

    def calculate_similarity_matrix(self):
        question_text = self.df['question'].fillna("").astype(str).tolist()
        question_vectors = TfidfVectorizer(max_features=1000, stop_words='english').fit_transform(question_text)
        self.similarity_matrix = cosine_similarity(question_vectors)

    def calculate_topic_matrix(self):
        question_text = self.df['question'].fillna("").astype(str).tolist()
        lda = GibbsLDA(n_topics=4, n_iter=10)
        self.topic_matrix = lda.fit_transform(question_text)

    def calculate_potential_matrix(self):
        diff_map = {'easy': 1, 'medium': 2, 'hard': 3}
        self.df['difficulty_num'] = self.df['difficulty'].map(diff_map).fillna(2).astype(int)

        bins = np.linspace(0, 1, 5)
        self.df['accuracy_bin'] = np.digitize(self.df['accuracy_norm'], bins) - 1
        
        joint_prob = pd.crosstab(self.df['accuracy_bin'], self.df['difficulty_num'], normalize='all')
        self.potential_matrix = np.zeros((4, 3))

        for acc_bin in range(4):
            for diff in range(1, 4):
                if diff in joint_prob.columns and acc_bin in joint_prob.index:
                    self.potential_matrix[acc_bin, diff - 1] = joint_prob.loc[acc_bin, diff]
                else:
                    self.potential_matrix[acc_bin, diff - 1] = 0.01

    def topic_overlap(self, doc1_topics, doc2_topics):
        return np.sum(np.minimum(doc1_topics, doc2_topics))

    def build_graph(self):
        self.G = nx.Graph()
        for idx, row in self.df.iterrows():
            self.G.add_node(
                row['titleSlug'], 
                title=row['title'],
                difficulty=row['difficulty'], 
                likability=row['likability'], 
                accuracy=row['accuracy']
            )

        n_problems = len(self.df)
        for i in range(n_problems):
            title_i = self.df.iloc[i]['titleSlug']
            sim_indices = np.argsort(self.similarity_matrix[i])[-21:-1]
            for j in sim_indices:
                title_j = self.df.iloc[j]['titleSlug']
                content_similarity = self.similarity_matrix[i, j]
                topic_overlap_score = self.topic_overlap(self.topic_matrix[i], self.topic_matrix[j])
                combined_weight = (content_similarity + topic_overlap_score) / 2.0
                
                if self.G.has_edge(title_i, title_j):
                    continue
                self.G.add_edge(title_i, title_j, weight=combined_weight)

    def recommend_questions(self, solved_questions, top_n=3):
        recommendations = {}
        for solved in solved_questions:
            if solved not in self.G:
                continue
            neighbors = self.G[solved]
            for neighbor in neighbors:
                if neighbor not in solved_questions:
                    weight = neighbors[neighbor]['weight']
                    recommendations[neighbor] = max(recommendations.get(neighbor, 0.0), weight)
        return sorted(recommendations.items(), key=lambda x: x[1], reverse=True)[:top_n]

    def analyze_weak_areas(self, solved_slugs):
        target_topics = ["array", "string", "dynamic-programming", "hash-table", "sorting", "greedy", "depth-first-search", "binary-search", "tree", "breadth-first-search", "graph", "two-pointers", "matrix", "stack"]
        total_by_topic = {}
        for idx, row in self.df.iterrows():
            topics = row.get('topics')
            if isinstance(topics, list):
                for t in topics:
                    t_lower = str(t).lower()
                    if t_lower in target_topics:
                        total_by_topic[t_lower] = total_by_topic.get(t_lower, 0) + 1

        solved_by_topic = {t: 0 for t in target_topics}
        solved_df = self.df[self.df['titleSlug'].isin(solved_slugs)]
        for idx, row in solved_df.iterrows():
            topics = row.get('topics')
            if isinstance(topics, list):
                for t in topics:
                    t_lower = str(t).lower()
                    if t_lower in solved_by_topic:
                        solved_by_topic[t_lower] += 1

        analysis = []
        for t in target_topics:
            total = total_by_topic.get(t, 0)
            solved_count = solved_by_topic.get(t, 0)
            ratio = (solved_count / total) if total > 0 else 0.0
            
            if solved_count == 0:
                status = "Critical (Unsolved)"
                priority = 3
            elif solved_count < 2:
                status = "Needs Practice"
                priority = 2
            elif ratio < 0.08:
                status = "Improving"
                priority = 1
            else:
                status = "Proficient"
                priority = 0

            analysis.append({
                "topic": t.replace("-", " ").title(),
                "slug": t,
                "solved": solved_count,
                "total": total,
                "ratio": round(ratio * 100, 1),
                "status": status,
                "priority": priority
            })
        analysis.sort(key=lambda x: x['topic'])
        return analysis

    def recommend_questions_enhanced(self, solved_slugs, top_n=10, mode="balanced"):
        if not solved_slugs:
            starters = self.df[self.df['difficulty'] == 'easy'].sort_values(
                by=['likability', 'accuracy'], ascending=[False, False]
            )
            return [(row['titleSlug'], 1.0) for idx, row in starters.head(top_n).iterrows()]

        similar_recs = self.recommend_questions(solved_slugs, top_n=top_n * 3)
        similar_dict = {slug: score for slug, score in similar_recs}

        if mode == "similar":
            if not similar_recs:
                starters = self.df[self.df['difficulty'] == 'easy'].sort_values(by=['likability'], ascending=False)
                return [(row['titleSlug'], 0.5) for idx, row in starters.head(top_n).iterrows() if row['titleSlug'] not in solved_slugs]
            return similar_recs[:top_n]

        weak_areas = self.analyze_weak_areas(solved_slugs)
        weak_slugs = [wa['slug'] for wa in weak_areas if wa['priority'] >= 2]

        n_problems = len(self.df)
        mrf_model = PairwiseMRF(n_problems)
        solved_indices = set(self.df[self.df['titleSlug'].isin(solved_slugs)].index)
        
        for idx, row in self.df.iterrows():
            if idx in solved_indices:
                mrf_model.set_unary(idx, 1000.0, 0.0)
            else:
                p1 = row['likability_norm'] * 0.5 + (1.0 - row['accuracy_norm']) * 0.3
                weak_boost = 0.0
                row_topics = [str(t).lower() for t in row.get('topics', [])]
                for ws in weak_slugs:
                    if ws in row_topics:
                        weak_boost += 0.4
                        break
                p1 += weak_boost
                p1 = min(0.99, max(0.01, p1))
                mrf_model.set_unary(idx, 1.0 - p1, p1)

        for i in range(n_problems):
            sim_indices = np.argsort(self.similarity_matrix[i])[-4:-1]
            for j in sim_indices:
                if i < j:
                    weight = self.similarity_matrix[i, j]
                    mrf_model.add_edge(i, j, weight)

        bp = BeliefPropagation(mrf_model)
        bp.run_iterations(max_iter=3)
        beliefs = bp.compute_beliefs()

        candidates = {}
        for idx, row in self.df.iterrows():
            slug = row['titleSlug']
            if idx in solved_indices:
                continue
            bp_score = beliefs[idx, 1]
            sim_score = similar_dict.get(slug, 0.0)
            if mode == "weak_areas":
                candidates[slug] = bp_score * 0.8 + sim_score * 0.2
            else:
                candidates[slug] = bp_score * 0.5 + sim_score * 0.5

        sorted_cands = sorted(candidates.items(), key=lambda x: x[1], reverse=True)
        if len(sorted_cands) < top_n:
            seen = set(c[0] for c in sorted_cands)
            for slug, score in similar_recs:
                if slug not in seen and len(sorted_cands) < top_n:
                    sorted_cands.append((slug, score))
                    seen.add(slug)
        return sorted_cands[:top_n]

    def save_recommender(self, file_path="recommender.pkl"):
        with open(file_path, 'wb') as f:
            pickle.dump({
                'df': self.df,
                'similarity_matrix': self.similarity_matrix,
                'topic_matrix': self.topic_matrix,
                'potential_matrix': self.potential_matrix,
                'graph': self.G,
            }, f)
        logger.info(f"Saved recommender pickle to {file_path}")

    @staticmethod
    def load_recommender(file_path="recommender.pkl"):
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            recommender = QuestionRecommender.__new__(QuestionRecommender)
            recommender.df = data['df']
            recommender.similarity_matrix = data['similarity_matrix']
            recommender.topic_matrix = data['topic_matrix']
            recommender.potential_matrix = data['potential_matrix']
            recommender.G = data['graph']
            return recommender

## 6. Helper function to initialize recommender model

Loads the compiled recommender pickle cache or auto-trains it.

In [ ]:
def get_recommender(pkl_path="recommender.pkl", json_path="data.json"):
    if os.path.exists(pkl_path):
        try:
            logger.info(f"Loading recommender from cache '{pkl_path}'...")
            return QuestionRecommender.load_recommender(pkl_path)
        except Exception as e:
            logger.error(f"Failed to load cached recommender: {e}. Retraining...")
            
    logger.info("Pickle not found or invalid. Auto-training QuestionRecommender model (this may take up to a minute)...")
    recommender = QuestionRecommender(json_path)
    try:
        recommender.save_recommender(pkl_path)
    except Exception as e:
        logger.error(f"Failed to cache recommender to {pkl_path}: {e}")
    return recommender